In [8]:
import os
import pandas as pd
import numpy as np
import time
import rmm
import cudf
import cupy as cp
from rmm.allocators.cupy import rmm_cupy_allocator
import anndata as an
import scanpy as sc
from scipy.io import mmread
from scipy.sparse import csr_matrix

# Enable `managed_memory`
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

# Load in processed data

In [4]:
%%time
tdata = sc.read_h5ad("/nfs/turbo/umms-indikar/shared/projects/HSC/data/datasets/ng_2024/iHSC.h5ad")
tdata

CPU times: user 43.4 s, sys: 9.6 s, total: 53 s
Wall time: 58.6 s


AnnData object with n_obs × n_vars = 252607 × 27946
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'line.ident', 'percent.mt', 'percent.ribo', 'percent.mitoribo', 'S.Score', 'G2M.Score', 'Phase', 'integrated_snn_res.0.5', 'seurat_clusters', 'integrated_snn_res.1', 'integrated_snn_res.0.8', 'integrated_snn_res.0.3', 'integrated_snn_res.0.4', 'fract.ident', 'induction.ident', 'celltype', 'RETA'
    var: 'name'
    obsm: 'X_pca', 'X_umap'

In [5]:
# Check if X is raw counts
print("adata.raw is None?", tdata.raw is None)

total_cells = tdata.n_obs

# Randomly sample 100 unique indices
sampled_indices = np.random.choice(total_cells, size=1000, replace=False)

ddata = tdata[sampled_indices, :].copy()

X = ddata.X.toarray() if hasattr(ddata.X, "toarray") else ddata.X
print(X.dtype)
print(np.min(X), np.max(X), np.mean(X))

adata.raw is None? True
float64
0.0 7.4907262027043675 0.11059825451040059


## Load in raw counts (from R)

In [2]:
%%time
fpath = "/scratch/indikar_root/indikar1/shared_data/hematokytos/new_processed/Ng_raw_counts.mtx"
cells_path = "/scratch/indikar_root/indikar1/shared_data/hematokytos/new_processed/Ng_cells.csv"
genes_path = "/scratch/indikar_root/indikar1/shared_data/hematokytos/new_processed/Ng_genes.csv"

ng_counts = mmread(fpath).tocsr().transpose()
ng_cells = pd.read_csv(cells_path, header=0).iloc[:,0].tolist()
ng_genes = pd.read_csv(genes_path, header=0).iloc[:,0].tolist()

ndata = an.AnnData(X=ng_counts)
ndata.obs_names = ng_cells
ndata.var_names = ng_genes

ndata

CPU times: user 1min 27s, sys: 23.6 s, total: 1min 51s
Wall time: 2min 15s


AnnData object with n_obs × n_vars = 252607 × 27946

In [3]:
# check if X is actually raw counts
print("adata.raw is None?", ndata.raw is None)

total_cells = ndata.n_obs

# Randomly sample 100 unique indices
sampled_indices = np.random.choice(total_cells, size=1000, replace=False)

ddata = ndata[sampled_indices, :].copy()

X = ddata.X.toarray() if hasattr(ddata.X, "toarray") else ddata.X
print(X.dtype)
print(np.min(X), np.max(X), np.mean(X))

adata.raw is None? True
int64
0 1396 0.3909991769841838


## Add raw counts to Ng data object

In [6]:
# Check if names/orders match:
print((tdata.obs_names == ndata.obs_names).all())  # Should be True
print((tdata.var_names == ndata.var_names).all())  # Should be True

tdata.layers['raw_counts'] = ndata.X.copy()

print(tdata.layers['raw_counts'].shape)
# Should be (252607, 27946)

# Check a few values
print(tdata.layers['raw_counts'][:5, :5])

tdata

True
True
(252607, 27946)
<Compressed Sparse Column sparse matrix of dtype 'int64'
	with 0 stored elements and shape (5, 5)>


AnnData object with n_obs × n_vars = 252607 × 27946
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'line.ident', 'percent.mt', 'percent.ribo', 'percent.mitoribo', 'S.Score', 'G2M.Score', 'Phase', 'integrated_snn_res.0.5', 'seurat_clusters', 'integrated_snn_res.1', 'integrated_snn_res.0.8', 'integrated_snn_res.0.3', 'integrated_snn_res.0.4', 'fract.ident', 'induction.ident', 'celltype', 'RETA'
    var: 'name'
    obsm: 'X_pca', 'X_umap'
    layers: 'raw_counts'

In [7]:
print(tdata.layers['raw_counts'].nnz) # Number of nonzero values in entire matrix

774053504


In [22]:
# # write out
# outpath = "/scratch/indikar_root/indikar1/shared_data/hematokytos/new_processed/Ng_2024_raw.h5ad"

# tdata.write(outpath)
# tdata

AnnData object with n_obs × n_vars = 252607 × 27946
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'line.ident', 'percent.mt', 'percent.ribo', 'percent.mitoribo', 'S.Score', 'G2M.Score', 'Phase', 'integrated_snn_res.0.5', 'seurat_clusters', 'integrated_snn_res.1', 'integrated_snn_res.0.8', 'integrated_snn_res.0.3', 'integrated_snn_res.0.4', 'fract.ident', 'induction.ident', 'celltype', 'RETA'
    var: 'name'
    obsm: 'X_pca', 'X_umap'
    layers: 'raw_counts'

In [ ]:
# file moved to Turbo: /nfs/turbo/umms-indikar/shared/projects/HSC/updated_adata/Ng_2024_raw.h5ad